# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

## Generation Task

Using the OpenAI SDK, please create a **structured output** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../02_activities/documents/managing_oneself.pdf")
docs = loader.load()

input_text = ""
for page in docs:
    input_text += page.page_content + "\n"

In [ ]:
from pydantic import BaseModel, Field
from openai import OpenAI
import os

class ArticleCore(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str

# ---------------------------
# Final Output Model ONLY
# ---------------------------
class ArticleSummary(BaseModel):
    Author: str = Field(description="Author(s) of the article")
    Title: str = Field(description="Title of the article")
    Relevance: str = Field(description="Why this article is relevant for children")
    Summary: str = Field(description="A concise summary no longer than 300 tokens")
    Tone: str = Field(description="The tone used to produce the summary")
    InputTokens: int = Field(description="Number of input tokens")
    OutputTokens: int = Field(description="Number of output tokens")

# ---------------------------
# Client
# ---------------------------
client = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

# ---------------------------
# Tone
# ---------------------------
TONE = "Simple English for young children with motivational tone."

# ---------------------------
# Instructions (Developer Prompt)
# ---------------------------
instructions = (
    "You are good at reading complicated documents and summarizing them in a simple English for childern between the age of 6 - 12."
    "You are asked to extract structured information and generate summaries strictly in {tone}. "
    "Ensure all fields are correctly populated.  If fileds are not in, just say so. Do not hallucinate unknown values."
).format(tone=TONE)

# ---------------------------
# User Prompt (Dynamic Context)
# ---------------------------
user_prompt = """
Analyze the following document and produce a structured output.

<document>
{input_doc}
</document>

Requirements:
1. Extract Author and Title (if not available, return "Unknown").
2. Write a one-paragraph that explains why is this article is important to young growing minds at early age.
3. Write a concise summary (max 300 tokens).
4. Tone must strictly be "{tone}".
5. Return valid JSON only.
"""

# ---------------------------
# API Call (Parse ONLY core fields)
# ---------------------------
response = client.responses.parse(
    model="gpt-4o-mini",
    instructions=instructions,
    input=[
        {"role": "user", "content": user_prompt.format(input_doc=input_text, tone=TONE)}
    ],
    # IMPORTANT: exclude token fields here
    text_format=ArticleCore,
    max_output_tokens=300
)

# ---------------------------
# Parsed Output
# ---------------------------
parsed = response.output_parsed

# ---------------------------
# Final Output (Add tokens cleanly)
# ---------------------------
result = ArticleSummary(
    **response.output_parsed.model_dump(),
    InputTokens=response.usage.input_tokens,
    OutputTokens=response.usage.output_tokens
)


In [4]:
print(result.model_dump_json(indent=2))

{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "This article teaches kids about being responsible for their own growth and success. It helps them understand their strengths and how to work well with others, which is very important as they grow up and choose careers. Learning these skills early can make them confident and successful in anything they want to pursue.",
  "Summary": "In this article, Peter Drucker tells us that we must know ourselves to be successful. It's like being your own boss! First, we should find out our strengths and weaknesses. To do this, we can use feedback, which means checking if our decisions worked out like we thought. Next, we need to learn how we best perform—some people learn best by reading, while others learn by listening or doing. We also need to know our values, which are the things that matter to us, like honesty and kindness. Finally, it’s important to make sure our jobs match who we are, so we can shine bright in ou

In [5]:
from IPython.display import display, Markdown

display(Markdown(f"### {result.Title}"))
display(Markdown(f"**Author:** {result.Author}"))
display(Markdown(f"**Tone:** {result.Tone}"))
display(Markdown(f"**Input Tokens:** {result.InputTokens} | **Output Tokens:** {result.OutputTokens}"))
display(Markdown(f"#### Relevance\n{result.Relevance}"))
display(Markdown(f"#### Summary\n{result.Summary}"))

### Managing Oneself

**Author:** Peter F. Drucker

**Tone:** Simple English for young children with motivational tone.

**Input Tokens:** 12395 | **Output Tokens:** 225

#### Relevance
This article teaches kids about being responsible for their own growth and success. It helps them understand their strengths and how to work well with others, which is very important as they grow up and choose careers. Learning these skills early can make them confident and successful in anything they want to pursue.

#### Summary
In this article, Peter Drucker tells us that we must know ourselves to be successful. It's like being your own boss! First, we should find out our strengths and weaknesses. To do this, we can use feedback, which means checking if our decisions worked out like we thought. Next, we need to learn how we best perform—some people learn best by reading, while others learn by listening or doing. We also need to know our values, which are the things that matter to us, like honesty and kindness. Finally, it’s important to make sure our jobs match who we are, so we can shine bright in our work and help others around us!

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [6]:
from deepeval.models import GPTModel
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

In [7]:
eval_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
)

In [8]:
summary_metric = SummarizationMetric(
    model=eval_model,
    threshold=0.8,
    assessment_questions=[
        "Does the summary capture the main idea?",
        "Does the summary uses the specified tone?",
        "Does the summary served the purpose?",
        "Does the summary factually consistent?",
        "Does the summary made up anything?"
    ]
)

# Coherence Metric
clarity_metrix = GEval(
    name="Clarity",
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
)

# Tonality Metric
professionalism = GEval(
    name="Professionalism",
    evaluation_steps=[
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
)


# Safety Metric 
pii_leakage = GEval(
    name="PII Leakage",
    evaluation_steps=[
        "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
        "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",
        "Ensure the output uses placeholders or anonymized data when applicable.",
        "Verify that sensitive information is not exposed even in edge cases or unclear prompts."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
)

# Create test case: input is the original document, actual_output is the generated summary
test_case = LLMTestCase(
    input=input_text,
    actual_output=result.Summary,
)

summary_metric.measure(test_case)
clarity_metrix.measure(test_case) # coherence metric
professionalism.measure(test_case) # tonality metric
pii_leakage.measure(test_case) # safety

# Structured evaluation output
class JsonResult(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

eval_result1 = JsonResult(
    SummarizationScore=summary_metric.score,
    SummarizationReason=summary_metric.reason,
    CoherenceScore=clarity_metrix.score,
    CoherenceReason=clarity_metrix.reason,
    TonalityScore=professionalism.score,
    TonalityReason=professionalism.reason,
    SafetyScore=pii_leakage.score,
    SafetyReason=pii_leakage.reason,
)


Output()

Output()

Output()

Output()

In [9]:
print(eval_result1.model_dump_json(indent=2))

{
  "SummarizationScore": 0.5714285714285714,
  "SummarizationReason": "The score is 0.57 because the summary contradicts the original text by suggesting a focus on both strengths and weaknesses, while the original emphasizes improving strengths only. Additionally, the summary includes extra information about feedback and learning styles that were not present in the original text, which detracts from its accuracy.",
  "CoherenceScore": 0.8320821300824607,
  "CoherenceReason": "The response uses clear and direct language, making the concepts accessible. It effectively avoids jargon and explains ideas like feedback and learning styles in simple terms. However, while the overall structure is easy to follow, some phrases like 'shine bright in our work' could be seen as slightly vague, which may detract from clarity.",
  "TonalityScore": 0.34212410248603187,
  "TonalityReason": "The response lacks a professional tone, using casual phrases like 'shine bright' and 'like being your own boss.' 

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [10]:
enhance_instructions = (
    "You are a document summarizer, especially for young children. "
    f"You must write all summaries using a {TONE} tone."
    "Use the previous summary evaluation and produce a new summary. Fix any issues in the previous summary. Increasing number of tokens, if needed." 
)

previous_summary = LLMTestCase.actual_output

improved_prompt = """"
"Rewrite the summary so it is:
1. More faithful to the context
2. Clear and concise
3. Factually accurate
4. Better aligned with the source text only

Use these feedback document to produce the improved summary and Return only the improved summary"
<document>
{input_doc}
</document>

<previous_summary>
{previous_summary}
</previous_summary>

<evaluation_feedback>
{eval_result}
</evaluation_feedback>

"""

enhance_response1 = client.responses.parse(
    model="gpt-4o-mini",
    instructions=instructions,
    input=[
        {"role": "user", "content": improved_prompt.format(input_doc=input_text, previous_summary=result.Summary, 
                                                       eval_result=eval_result1.model_dump_json(indent=2))}
    ],
    text_format=ArticleSummary,
    max_output_tokens=500
)

In [11]:
display(Markdown(f"## Enhanced Summary\n{enhance_response1.output_parsed.Summary}"))

## Enhanced Summary
In 'Managing Oneself,' Peter Drucker explains that success comes from knowing ourselves really well. This means understanding our strengths, how we work best, and what we believe in. To find out our strengths, we can look at past decisions to see what worked and what didn’t. Learning how we perform is also important; some people learn best by reading, while others might learn by listening or doing. Knowing what is important to us, like honesty, helps ensure we choose jobs that fit who we are. When we know ourselves, we can do great work and help others too!

In [12]:
enhanced_version = enhance_response1.output_parsed

# Re-evaluate the enhanced summary using the same metrics
enhanced_test_case = LLMTestCase(
    input=input_text,
    actual_output=enhanced_version.Summary,
)

summary_metric.measure(enhanced_test_case)
clarity_metrix.measure(enhanced_test_case)
professionalism.measure(enhanced_test_case)
pii_leakage.measure(enhanced_test_case)

eval_result2 = JsonResult(
    SummarizationScore=summary_metric.score,
    SummarizationReason=summary_metric.reason,
    CoherenceScore=clarity_metrix.score,
    CoherenceReason=clarity_metrix.reason,
    TonalityScore=professionalism.score,
    TonalityReason=professionalism.reason,
    SafetyScore=pii_leakage.score,
    SafetyReason=pii_leakage.reason,
)

Output()

Output()

Output()

Output()

In [13]:
enhance_response2 = eval_result2.model_dump_json(indent=2)
print(enhance_response2)

{
  "SummarizationScore": 0.6666666666666666,
  "SummarizationReason": "The score is 0.67 because the summary includes extra information that was not present in the original text, which may lead to misinterpretation of the original message. However, there are no contradictions, indicating a reasonable alignment with the core ideas.",
  "CoherenceScore": 0.8731058578630003,
  "CoherenceReason": "The response uses clear and direct language, effectively conveying the main ideas from 'Managing Oneself' without jargon. It presents complex concepts, such as self-awareness and learning styles, in an easy-to-follow manner. However, while the explanation is mostly clear, the phrase 'do great work and help others too' could be seen as slightly vague, which prevents a perfect score.",
  "TonalityScore": 0.44654036429465166,
  "TonalityReason": "The response maintains a generally professional tone but lacks the level of expertise and formality expected in a domain-specific context. While it convey

In [16]:
print("Response 1")
print(eval_result1.model_dump_json(indent=2))
print("--------------------")
print("Response 2")
print(enhance_response2)

Response 1
{
  "SummarizationScore": 0.5714285714285714,
  "SummarizationReason": "The score is 0.57 because the summary contradicts the original text by suggesting a focus on both strengths and weaknesses, while the original emphasizes improving strengths only. Additionally, the summary includes extra information about feedback and learning styles that were not present in the original text, which detracts from its accuracy.",
  "CoherenceScore": 0.8320821300824607,
  "CoherenceReason": "The response uses clear and direct language, making the concepts accessible. It effectively avoids jargon and explains ideas like feedback and learning styles in simple terms. However, while the overall structure is easy to follow, some phrases like 'shine bright in our work' could be seen as slightly vague, which may detract from clarity.",
  "TonalityScore": 0.34212410248603187,
  "TonalityReason": "The response lacks a professional tone, using casual phrases like 'shine bright' and 'like being your 

Please, do not forget to add your comments.

The second version was better. The score improved from 0.57 to 0.66, which shows that using evaluation feedback helped improve the summary. It became more aligned with the original text and reduced some of the earlier mistakes.

However, it is still was not perfect. Some important ideas were still missing, and a few parts were slightly misleading. So while this feedback loop definitely helps, it is not enough on its own.

In a real system, I would also add better document retrieval, clearer prompts, multiple evaluation checks, and human review when accuracy really matters.

Overall, this exercise shows that AI outputs can improve through self-correction, but quality control should use more than just one method.  Since I the number of tokens are reduced due to quota limit, that impacts the results.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
